# Week 07 — Room Database Concepts and SQL Fundamentals
## LeafGuard AI: Understanding Local Data Persistence

Room database is an Android Java component, but the underlying concepts (SQL, relational data)
are best learned with Python's sqlite3 module — same SQL, faster feedback loop.

**Learning outcomes:**
- Understand SQLite's role in Android persistence
- Write SQL CREATE, INSERT, SELECT, UPDATE, DELETE statements
- Design the `scan_records` table schema
- Understand how Room annotations map to SQL
- Practice queries matching LeafGuard AI's DAO methods

---

## Cell 1: SQLite in Python vs SQLite in Android

Python's `sqlite3` module uses the **same SQLite engine** as Android Room.
The SQL syntax is identical. Only the API differs.

```
Python sqlite3              Android Room
───────────────             ────────────────────
conn.cursor()           ←→  AppDatabase.getInstance()
cursor.execute(sql)     ←→  @Query("SELECT ...")
cursor.fetchall()       ←→  dao.getAllScans()
conn.commit()           ←→  Auto-committed by Room
```

In [ ]:
import sqlite3
import os

# Python's sqlite3 is built-in — no installation needed
print(f'sqlite3 module version: {sqlite3.version}')
print(f'SQLite library version: {sqlite3.sqlite_version}')
print()
print('This is the SAME SQLite that Android Room uses!')
print('The SQL queries you write here will work identically in your DAO.')

## Cell 2: Create the LeafGuard AI Database Schema

This matches the Room `@Entity` class `ScanRecord.java` exactly.

In [ ]:
import sqlite3

# Use in-memory database for this notebook (no file created)
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row  # Access columns by name
cursor = conn.cursor()

# Create table matching LeafGuard AI's ScanRecord @Entity
# This is the SQL that Room generates from the @Entity annotation
CREATE_TABLE_SQL = '''
CREATE TABLE IF NOT EXISTS scan_records (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    disease_name    TEXT        NOT NULL,
    confidence      REAL        NOT NULL,
    symptoms        TEXT,
    treatment       TEXT,
    image_uri       TEXT,
    timestamp       INTEGER     NOT NULL DEFAULT (strftime('%s', 'now') * 1000),
    is_cloud_scan   INTEGER     NOT NULL DEFAULT 1
);
'''

# This is the Room @Entity equivalent:
# @Entity(tableName = "scan_records")
# public class ScanRecord {
#     @PrimaryKey(autoGenerate = true) int id;
#     @ColumnInfo(name = "disease_name") String diseaseName;
#     @ColumnInfo(name = "confidence")   float  confidence;
#     ... etc
# }

cursor.execute(CREATE_TABLE_SQL)
conn.commit()

# Verify the table was created
cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print('Tables created:')
for table in tables:
    print(f'  Table: {table["name"]}')
    print(f'  SQL: {table["sql"]}')

# Show column info
cursor.execute('PRAGMA table_info(scan_records)')
columns = cursor.fetchall()
print()
print('Column definitions:')
print(f'{"#":<4} {"Name":<20} {"Type":<10} {"NotNull":<10} {"PK":<5}')
print('-' * 55)
for col in columns:
    print(f'{col["cid"]:<4} {col["name"]:<20} {col["type"]:<10} {col["notnull"]:<10} {col["pk"]:<5}')

## Cell 3: INSERT — Saving Scan Results

Corresponds to `@Insert void insert(ScanRecord record)` in `ScanDao.java`

In [ ]:
import time

def insert_scan(cursor, conn, disease_name, confidence, symptoms, treatment, image_uri=None, is_cloud=True):
    """Insert a new scan record. Returns the row ID."""
    cursor.execute(
        '''
        INSERT INTO scan_records (disease_name, confidence, symptoms, treatment, image_uri, timestamp, is_cloud_scan)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ''',
        (disease_name, confidence, symptoms, treatment, image_uri,
         int(time.time() * 1000),  # Unix milliseconds
         1 if is_cloud else 0)
    )
    conn.commit()
    return cursor.lastrowid

# Room Java equivalent:
# @Insert(onConflict = OnConflictStrategy.REPLACE)
# void insert(ScanRecord record);

# Insert sample scan records
sample_scans = [
    ('Tomato Early Blight', 0.91, 'Brown concentric-ring lesions on older leaves', 'Apply copper fungicide', None, True),
    ('Tomato Healthy', 0.95, 'No visible disease symptoms', 'No treatment needed', None, True),
    ('Corn Common Rust', 0.88, 'Small, circular reddish-brown pustules on leaves', 'Apply fungicide at first sign', None, False),
    ('Apple Scab', 0.73, 'Olive-green spots on leaves, dark scabby fruit', 'Remove infected leaves, apply lime sulfur', None, True),
    ('Grape Black Rot', 0.82, 'Circular brown lesions with black dots (pycnidia)', 'Remove mummified fruit, apply mancozeb', None, True),
    ('Tomato Late Blight', 0.96, 'Irregular greenish-grey water-soaked lesions', 'Remove infected plants immediately, apply metalaxyl', None, True),
]

inserted_ids = []
for scan in sample_scans:
    row_id = insert_scan(cursor, conn, *scan)
    inserted_ids.append(row_id)

print(f'Inserted {len(inserted_ids)} scan records')
print(f'Row IDs: {inserted_ids}')
print()

# Verify count
cursor.execute('SELECT COUNT(*) as count FROM scan_records')
count = cursor.fetchone()['count']
print(f'Total records in database: {count}')
print()
print('Room Java equivalent:')
print('''
// In your Activity/AsyncTask:
ScanRecord record = new ScanRecord(
    "Tomato Early Blight", 0.91f,
    "Brown concentric-ring lesions...",
    "Apply copper fungicide",
    imageUri.toString(), true
);
AppDatabase.getInstance(context).scanDao().insert(record);
''')

## Cell 4: SELECT — Querying Scan History

In [ ]:
from datetime import datetime

def format_ts(ts_ms):
    return datetime.fromtimestamp(ts_ms / 1000).strftime('%d %b %Y %H:%M')

print('=== SELECT QUERIES ===')
print()

# 1. Get all scans, newest first
# Room: @Query("SELECT * FROM scan_records ORDER BY timestamp DESC")
cursor.execute('SELECT * FROM scan_records ORDER BY timestamp DESC')
all_scans = cursor.fetchall()
print(f'1. All scans ({len(all_scans)} total), newest first:')
for scan in all_scans:
    print(f'   [{scan["id"]}] {scan["disease_name"]:<35} {scan["confidence"]*100:.0f}% | {format_ts(scan["timestamp"])}')

print()

# 2. Get recent N scans
# Room: @Query("SELECT * FROM scan_records ORDER BY timestamp DESC LIMIT :limit")
cursor.execute('SELECT * FROM scan_records ORDER BY timestamp DESC LIMIT ?', (3,))
recent = cursor.fetchall()
print(f'2. Most recent 3 scans:')
for scan in recent:
    print(f'   [{scan["id"]}] {scan["disease_name"]}')

print()

# 3. Search by disease name (partial match)
# Room: @Query("SELECT * FROM scan_records WHERE disease_name LIKE '%' || :query || '%'")
cursor.execute("SELECT * FROM scan_records WHERE disease_name LIKE ?", ('%Tomato%',))
tomato_scans = cursor.fetchall()
print(f'3. Scans matching "Tomato" ({len(tomato_scans)} results):')
for scan in tomato_scans:
    print(f'   {scan["disease_name"]}')

print()

# 4. Count total scans
# Room: @Query("SELECT COUNT(*) FROM scan_records")
cursor.execute('SELECT COUNT(*) as total, AVG(confidence) as avg_conf FROM scan_records')
stats = cursor.fetchone()
print(f'4. Statistics: total={stats["total"]}, avg confidence={stats["avg_conf"]*100:.1f}%')

print()

# 5. Get by ID
# Room: @Query("SELECT * FROM scan_records WHERE id = :id")
cursor.execute('SELECT * FROM scan_records WHERE id = ?', (1,))
record = cursor.fetchone()
print(f'5. Scan #1: {record["disease_name"]} | treatment: {record["treatment"]}')

## Cell 5: UPDATE and DELETE Operations

In [ ]:
print('=== UPDATE OPERATIONS ===')
print()

# Show before
cursor.execute('SELECT disease_name, treatment FROM scan_records WHERE id = 1')
before = cursor.fetchone()
print(f'Before update: treatment = "{before["treatment"][:50]}..."')

# UPDATE: Modify a treatment
# Room: @Query("UPDATE scan_records SET treatment = :treatment WHERE id = :id")
cursor.execute(
    'UPDATE scan_records SET treatment = ? WHERE id = ?',
    ('Prune infected leaves. Apply Bordeaux mixture every 7-10 days.', 1)
)
conn.commit()

cursor.execute('SELECT disease_name, treatment FROM scan_records WHERE id = 1')
after = cursor.fetchone()
print(f'After update:  treatment = "{after["treatment"][:60]}"')
print()

print('=== DELETE OPERATIONS ===')
print()

# Count before
cursor.execute('SELECT COUNT(*) as cnt FROM scan_records')
before_count = cursor.fetchone()['cnt']
print(f'Records before delete: {before_count}')

# DELETE by ID
# Room: @Query("DELETE FROM scan_records WHERE id = :id")
cursor.execute('DELETE FROM scan_records WHERE id = ?', (4,))  # Delete Apple Scab
conn.commit()

cursor.execute('SELECT COUNT(*) as cnt FROM scan_records')
after_count = cursor.fetchone()['cnt']
print(f'Records after delete:  {after_count}')
print(f'Deleted: {before_count - after_count} record(s)')
print()

# DELETE by condition (old scans)
import time
cutoff = int(time.time() * 1000) - (30 * 24 * 60 * 60 * 1000)  # 30 days ago
cursor.execute('SELECT COUNT(*) as cnt FROM scan_records WHERE timestamp < ?', (cutoff,))
old_count = cursor.fetchone()['cnt']
print(f'Scans older than 30 days: {old_count}')
print()
print('Room DAO equivalent:')
print('@Query("DELETE FROM scan_records WHERE timestamp < :cutoffTime")')
print('void deleteOlderThan(long cutoffTime);')

## Cell 6: Aggregation and Analytics Queries

In [ ]:
print('=== AGGREGATION QUERIES ===')
print()

# Disease frequency analysis
cursor.execute('''
    SELECT
        disease_name,
        COUNT(*) as scan_count,
        AVG(confidence) as avg_confidence,
        MAX(confidence) as max_confidence,
        MIN(timestamp) as first_seen
    FROM scan_records
    GROUP BY disease_name
    ORDER BY scan_count DESC
''')
frequency = cursor.fetchall()

print('Disease occurrence frequency:')
print(f'{"Disease":<35} {"Count":>6} {"Avg Conf":>9} {"Max Conf":>9}')
print('-' * 65)
for row in frequency:
    print(f'{row["disease_name"]:<35} {row["scan_count"]:>6} '
          f'{row["avg_confidence"]*100:>8.1f}% {row["max_confidence"]*100:>8.1f}%')

print()

# Separate healthy vs diseased count
cursor.execute('''
    SELECT
        CASE WHEN disease_name LIKE '%Healthy%' OR disease_name LIKE '%healthy%'
             THEN 'Healthy' ELSE 'Diseased' END as status,
        COUNT(*) as count,
        ROUND(AVG(confidence) * 100, 1) as avg_confidence_pct
    FROM scan_records
    GROUP BY status
''')
health_stats = cursor.fetchall()
print('Health status breakdown:')
total = sum(row['count'] for row in health_stats)
for row in health_stats:
    pct = row['count'] / total * 100
    bar = '█' * int(pct / 5)
    print(f'  {row["status"]:<10}: {row["count"]:>3} ({pct:>5.1f}%) {bar} (avg confidence: {row["avg_confidence_pct"]}%)')

print()

# Cloud vs offline scan breakdown
cursor.execute('''
    SELECT
        CASE WHEN is_cloud_scan = 1 THEN 'Cloud (FastAPI)' ELSE 'Offline (TFLite)' END as mode,
        COUNT(*) as count,
        ROUND(AVG(confidence) * 100, 1) as avg_conf
    FROM scan_records
    GROUP BY is_cloud_scan
''')
mode_stats = cursor.fetchall()
print('Detection mode breakdown:')
for row in mode_stats:
    print(f'  {row["mode"]:<20}: {row["count"]} scans, avg confidence {row["avg_conf"]}%')

## Cell 7: Room vs Raw SQLite — Side-by-Side Comparison

In [ ]:
print('=== ROOM vs RAW SQLITE: Side-by-Side Comparison ===')
print()

comparisons = [
    {
        'operation': 'INSERT one record',
        'raw_sqlite': '''
// Raw SQLite (verbose, error-prone)
SQLiteDatabase db = helper.getWritableDatabase();
ContentValues values = new ContentValues();
values.put("disease_name", record.getDiseaseName());
values.put("confidence", record.getConfidence());
values.put("timestamp", record.getTimestamp());
long id = db.insert("scan_records", null, values);
db.close();
        ''',
        'room': '''
// Room (clean, type-safe)
@Insert
void insert(ScanRecord record);

// Usage:
db.scanDao().insert(record);
        '''
    },
    {
        'operation': 'SELECT all, newest first',
        'raw_sqlite': '''
// Raw SQLite (manual cursor parsing)
Cursor cursor = db.rawQuery(
    "SELECT * FROM scan_records ORDER BY timestamp DESC", null
);
List<ScanRecord> results = new ArrayList<>();
while (cursor.moveToNext()) {
    ScanRecord r = new ScanRecord();
    r.setId(cursor.getInt(cursor.getColumnIndex("id")));
    r.setDiseaseName(cursor.getString(cursor.getColumnIndex("disease_name")));
    r.setConfidence(cursor.getFloat(cursor.getColumnIndex("confidence")));
    // ... more fields ...
    results.add(r);
}
cursor.close();
        ''',
        'room': '''
// Room (zero boilerplate)
@Query("SELECT * FROM scan_records ORDER BY timestamp DESC")
List<ScanRecord> getAllScans();

// Usage:
List<ScanRecord> scans = db.scanDao().getAllScans();
        '''
    },
    {
        'operation': 'DELETE by ID',
        'raw_sqlite': '''
// Raw SQLite
db.delete("scan_records", "id = ?", new String[]{String.valueOf(id)});
        ''',
        'room': '''
// Room
@Query("DELETE FROM scan_records WHERE id = :id")
void deleteById(int id);

// Usage:
db.scanDao().deleteById(scanId);
        '''
    }
]

for comp in comparisons:
    print(f'OPERATION: {comp["operation"]}')
    print()
    print('  Raw SQLite:', comp['raw_sqlite'])
    print('  Room:', comp['room'])
    print('─' * 70)

## Cell 8: Database Migration Simulation

In [ ]:
import sqlite3

print('=== DATABASE MIGRATION SIMULATION ===')
print()

# Simulate schema version management
# Create version 1 database
conn_v1 = sqlite3.connect(':memory:')
conn_v1.row_factory = sqlite3.Row
c = conn_v1.cursor()

# Version 1 schema (original)
c.execute('''
CREATE TABLE scan_records (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    disease_name TEXT NOT NULL,
    confidence REAL NOT NULL,
    timestamp INTEGER NOT NULL
)
''')
conn_v1.commit()

# Insert some v1 data
import time
c.execute('INSERT INTO scan_records (disease_name, confidence, timestamp) VALUES (?, ?, ?)',
          ('Tomato Early Blight', 0.91, int(time.time() * 1000)))
c.execute('INSERT INTO scan_records (disease_name, confidence, timestamp) VALUES (?, ?, ?)',
          ('Tomato Healthy', 0.95, int(time.time() * 1000)))
conn_v1.commit()

c.execute('SELECT COUNT(*) as cnt FROM scan_records')
print(f'Version 1 database: {c.fetchone()["cnt"]} records')

# Show v1 columns
c.execute('PRAGMA table_info(scan_records)')
v1_cols = [col['name'] for col in c.fetchall()]
print(f'V1 columns: {v1_cols}')

print()
print('=== MIGRATION 1 → 2: Add symptoms and treatment columns ===')

# MIGRATION 1 → 2
# In Room Java:
# static final Migration MIGRATION_1_2 = new Migration(1, 2) {
#     @Override
#     public void migrate(SupportSQLiteDatabase database) {
#         database.execSQL("ALTER TABLE scan_records ADD COLUMN symptoms TEXT");
#         database.execSQL("ALTER TABLE scan_records ADD COLUMN treatment TEXT");
#     }
# };

c.execute('ALTER TABLE scan_records ADD COLUMN symptoms TEXT')
c.execute('ALTER TABLE scan_records ADD COLUMN treatment TEXT')
conn_v1.commit()

# Verify migration preserved existing data
c.execute('SELECT * FROM scan_records')
all_records = c.fetchall()
print(f'Records after migration: {len(all_records)} (preserved!)')

c.execute('PRAGMA table_info(scan_records)')
v2_cols = [col['name'] for col in c.fetchall()]
print(f'V2 columns: {v2_cols}')

print()
print('IMPORTANT: Always write migrations. Never use fallbackToDestructiveMigration() in production!')
print('If you do, ALL user scan history is deleted when the app updates!')

conn_v1.close()

## Cell 9: Background Threading Concept

Android Room enforces background thread operations. This cell shows WHY that matters.

In [ ]:
import time
import threading

print('=== WHY BACKGROUND THREADING MATTERS ===')
print()

# Simulate database operation timing
def simulate_db_on_main_thread():
    """This is what happens if you call Room on the main (UI) thread."""
    print('[MAIN THREAD] User taps "Detect" button')
    print('[MAIN THREAD] UI is FROZEN while waiting for database...')

    # Simulate slow database operation
    start = time.time()
    # Real database insert with filesystem I/O can take 50-500ms
    time.sleep(0.3)  # 300ms simulated DB operation
    elapsed = (time.time() - start) * 1000

    print(f'[MAIN THREAD] Database done after {elapsed:.0f}ms')
    print('[MAIN THREAD] UI UNFROZEN — but user saw janky freeze!')
    print('[MAIN THREAD] Android may show ANR (App Not Responding) dialog!')

print('WRONG (main thread — freezes UI):')
simulate_db_on_main_thread()

print()
print('-' * 50)
print()

ui_thread_done = threading.Event()
db_result = []

def simulate_db_on_background_thread():
    """This is what Android Thread or Room @Query does on background."""
    # Background thread does the heavy work
    time.sleep(0.3)  # 300ms DB operation happens in background
    db_result.append('Scan records loaded!')
    # Signals the UI to update
    print('[BACKGROUND] DB complete — posting result to UI thread')
    ui_thread_done.set()

print('CORRECT (background thread — UI stays responsive):')
print('[MAIN THREAD] User taps button')
print('[MAIN THREAD] Start background thread for DB operation')

t = threading.Thread(target=simulate_db_on_background_thread)
t.start()

# Main thread continues immediately
print('[MAIN THREAD] Showing loading spinner to user...')
print('[MAIN THREAD] UI is responsive — user can still scroll, rotate, etc.')

ui_thread_done.wait()  # Wait for background thread
print(f'[MAIN THREAD] Received: "{db_result[0]}" → update RecyclerView')

print()
print('Android Room Java equivalent:')
print('''
new Thread(() -> {
    // Background thread: safe to call Room here
    List<ScanRecord> records = db.scanDao().getAllScans();

    // Post to main thread to update UI
    runOnUiThread(() -> {
        adapter.submitList(records);
        progressBar.setVisibility(View.GONE);
    });
}).start();
''')

## Cell 10: Checkpoint Exercises

1. ☐ Add a `prevention TEXT` column to the table and insert data with it
2. ☐ Write a query to find the most common disease detected
3. ☐ Write a query to find all scans where confidence < 0.6 (uncertain)
4. ☐ Simulate migration from v2 (adding `prevention`) to v3 (adding `image_uri`)
5. ☐ Write a query counting scans per day (group by date from timestamp)
6. ☐ In the Android project, open `ScanDao.java` — map each method to its equivalent SQL query

**Bonus:** Write a query that returns the "scan of the day" — the detection with highest confidence from the current day.

## Summary

| SQL Concept | Room Annotation | Example |
|------------|----------------|--------|
| CREATE TABLE | `@Entity` | `@Entity(tableName="scan_records")` |
| INSERT | `@Insert` | `void insert(ScanRecord r)` |
| SELECT | `@Query` | `@Query("SELECT * FROM scan_records")` |
| UPDATE | `@Update` / `@Query` | `@Update void update(ScanRecord r)` |
| DELETE | `@Delete` / `@Query` | `@Delete void delete(ScanRecord r)` |
| PRIMARY KEY | `@PrimaryKey` | `@PrimaryKey(autoGenerate=true)` |
| Column name | `@ColumnInfo` | `@ColumnInfo(name="disease_name")` |

---
*See also: `roadmap/week-07-room-sqlite-history/learning-notes.md`*  
*And: `android-app/app/src/main/java/com/leafguard/database/`*